# Projeto Final — Análise de Vendas de Supermercado

## Storytelling com dados, estatística e tomada de decisão

### Contexto de negócio

Uma base de produtos de supermercado pode revelar muito mais do que preços e descontos. Quando essas informações são organizadas e analisadas estatisticamente, é possível identificar padrões de preço, heterogeneidade entre categorias, valores extremos e diferentes estratégias promocionais.

Neste projeto, a análise será conduzida como uma investigação de negócio:

> **O que os números estão dizendo sobre preços, descontos e oportunidades comerciais?**

O objetivo não é apenas calcular métricas, mas **interpretar o comportamento dessas métricas e transformá-lo em hipóteses úteis para a tomada de decisão**.

## Perguntas de negócio

A análise será guiada pelas seguintes perguntas:

1. Quais categorias possuem os maiores preços médios e medianos?
2. Onde a diferença entre média e mediana sugere maior assimetria?
3. Qual categoria apresenta maior variabilidade de preços?
4. Os valores extremos representam possíveis erros ou produtos com características comerciais diferentes?
5. Quais categorias concentram os maiores descontos médios?
6. Quais combinações de categoria e marca merecem investigação comercial?
7. Que decisões ou perguntas adicionais podem ser derivadas desses resultados?

A lógica do projeto será:

**Dados → Métricas → Comportamento → Hipóteses → Insight de negócio → Próxima ação**

## 1. Importação das bibliotecas

Serão utilizadas bibliotecas de manipulação de dados, estatística e visualização.

- **Pandas:** organização e agregação dos dados;
- **NumPy:** operações numéricas e classificação das métricas;
- **Matplotlib e Seaborn:** gráficos estatísticos;
- **Plotly:** visualização interativa.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Carregamento da base

A base será carregada em um DataFrame. A primeira preocupação é garantir que o conjunto de dados esteja disponível e que sua estrutura seja conhecida antes de qualquer interpretação.

Uma análise de negócio só é confiável quando as métricas são calculadas sobre dados que foram previamente inspecionados.

In [ ]:
arquivo = "MODULO7_PROJETOFINAL_BASE_SUPERMERCADO - MODULO7_PROJETOFINAL_BASE_SUPERMERCADO (1).csv.csv"

df = pd.read_csv(arquivo)

print("Base carregada com sucesso!")
print(f"Quantidade de registros: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

display(df.head())

## 3. Auditoria inicial dos dados

Antes de procurar padrões, precisamos verificar a qualidade da matéria-prima.

Valores ausentes podem reduzir a quantidade de observações utilizadas nas métricas. Duplicidades podem superdimensionar determinados produtos. Tipos de dados inadequados podem impedir cálculos corretos.

Esta etapa não é apenas técnica: **qualidade dos dados influencia diretamente a qualidade das decisões**.

In [ ]:
print("Dimensões:", df.shape)

print("\nColunas:")
print(df.columns.tolist())

print("\nTipos de dados:")
display(df.dtypes.to_frame("Tipo"))

print("\nValores nulos:")
display(df.isnull().sum().to_frame("Nulos"))

print("\nDuplicidades:")
print(df.duplicated().sum())

## 4. Perfil estatístico das categorias

Agora iniciamos a investigação do comportamento dos preços.

A **média** responde: *qual é o preço médio observado?*

A **mediana** responde: *qual é o preço que ocupa a posição central quando os produtos são ordenados?*

O **desvio padrão** responde: *quanto os preços variam em torno da média?*

Essas três medidas contam histórias diferentes. Uma categoria pode ter média alta porque possui alguns produtos muito caros, enquanto a mediana permanece relativamente baixa. Nesse caso, olhar somente para a média pode produzir uma interpretação incompleta.

In [ ]:
estatisticas_categoria = (
    df.groupby("Categoria")["Preco_Normal"]
      .agg(
          Media="mean",
          Mediana="median",
          Desvio_Padrao="std",
          Quantidade="count"
      )
      .reset_index()
)

estatisticas_categoria["Razao_Media_Mediana"] = (
    estatisticas_categoria["Media"] /
    estatisticas_categoria["Mediana"]
)

estatisticas_categoria["CV"] = (
    estatisticas_categoria["Desvio_Padrao"] /
    estatisticas_categoria["Media"] * 100
)

display(
    estatisticas_categoria
    .round(2)
    .sort_values("Media", ascending=False)
)

## 5. Leitura de negócio: média × mediana

A categoria **comidas-preparadas** apresenta a maior média de preço, aproximadamente **3,095.04**. Isso significa que, considerando todos os produtos dessa categoria, o preço médio é superior ao das demais.

Entretanto, média alta não significa necessariamente que o produto "típico" daquela categoria seja caro. Para isso, a mediana é importante.

A maior mediana pertence a **comidas-preparadas**, com aproximadamente **3,290.00**.

### O que a diferença pode significar?

Quando a média fica muito acima da mediana, alguns preços elevados estão puxando o valor médio para cima. Esse comportamento pode indicar:

- presença de produtos premium;
- produtos de maior volume ou embalagem;
- marcas posicionadas em faixas de preço superiores;
- diferenças de mix dentro da categoria;
- possíveis registros extremos que merecem investigação.

Na perspectiva de negócio, isso significa que **a média deve ser interpretada junto com a mediana**, especialmente quando a empresa pretende definir preço de referência, segmentar produtos ou comparar categorias.

### Indicador adicional: razão média/mediana

Uma forma simples de tornar essa comparação mais objetiva é calcular:

**Razão = Média ÷ Mediana**

Quanto mais essa razão se afasta de 1, maior é a diferença relativa entre as duas medidas.

Isso não prova, isoladamente, a existência de uma distribuição assimétrica, mas funciona como um **sinal de alerta para investigação**.

In [ ]:
estatisticas_interpretacao = (
    estatisticas_categoria[
        [
            "Categoria",
            "Media",
            "Mediana",
            "Razao_Media_Mediana",
            "Desvio_Padrao",
            "CV"
        ]
    ]
    .sort_values("Razao_Media_Mediana", ascending=False)
)

display(estatisticas_interpretacao.round(2))

### Insight de assimetria

A maior razão entre média e mediana aparece em **lacteos**, com uma razão aproximada de **2.41**.

Em termos práticos, a média dessa categoria é aproximadamente **2.41 vezes** a sua mediana.

Esse resultado reforça a hipótese de que existe uma concentração de produtos em faixas de preço mais baixas acompanhada de alguns preços significativamente maiores.

**Implicação de negócio:** se uma empresa usar somente a média para definir uma referência de preço, pode superestimar o preço representativo dessa categoria. A mediana pode ser mais adequada quando o objetivo é representar o comportamento do produto típico.

## 6. Qual categoria apresenta maior risco de heterogeneidade de preços?

O desvio padrão mede a dispersão absoluta dos preços. Uma categoria com desvio padrão elevado possui preços mais espalhados.

Porém, existe uma ressalva importante: categorias com médias maiores naturalmente podem apresentar desvios padrão maiores.

Por isso, além do desvio padrão absoluto, calcularemos o **Coeficiente de Variação (CV)**:

**CV = Desvio padrão ÷ Média × 100**

O CV permite avaliar a dispersão de maneira relativa ao nível médio de preços.

In [ ]:
display(
    estatisticas_categoria[
        [
            "Categoria",
            "Media",
            "Desvio_Padrao",
            "CV"
        ]
    ]
    .sort_values("CV", ascending=False)
    .round(2)
)

### Leitura gerencial da dispersão

Pelo critério solicitado no desafio, a categoria com maior **desvio padrão absoluto** é **lacteos**, com aproximadamente **3,925.82**.

Já quando avaliamos a dispersão proporcional ao preço médio, o maior **coeficiente de variação** aparece em **lacteos**, com aproximadamente **164.59%**.

Essa distinção é importante:

- **Desvio padrão:** mostra a magnitude absoluta da variação.
- **CV:** mostra o tamanho da variação em relação ao preço médio.

### Implicação de negócio

Uma categoria muito heterogênea pode exigir uma estratégia de preço menos uniforme. Em vez de aplicar uma única regra de precificação, pode ser interessante segmentar os produtos por:

- marca;
- faixa de preço;
- posicionamento premium ou econômico;
- tamanho/embalagem;
- volume de vendas.

Assim, a estatística deixa de ser apenas descritiva e passa a sugerir **como a estratégia comercial pode ser segmentada**.

## 7. Boxplot da categoria com maior desvio padrão

O boxplot permite investigar visualmente a categoria selecionada.

A leitura deve considerar três elementos:

1. **Linha central:** mediana;
2. **Caixa:** intervalo entre primeiro e terceiro quartil;
3. **Pontos externos:** possíveis valores extremos.

O objetivo não é apenas "ver o gráfico", mas responder:

> **A elevada dispersão observada anteriormente é causada por uma distribuição naturalmente ampla ou por poucos valores extremos?**

In [ ]:
# ============================================================
# 5. BOXPLOT — DISTRIBUIÇÃO DE PREÇOS NA CATEGORIA MAIS DISPERSA
# ============================================================

plt.figure(figsize=(11, 6))

ax = sns.boxplot(
    data=df[df["Categoria"] == categoria_maior_desvio],
    x="Preco_Normal",
    color="#457B9D",
    width=0.45,
    flierprops={
        "marker": "o",
        "markerfacecolor": "#E76F51",
        "markeredgecolor": "#7A1F1F",
        "markersize": 5,
        "alpha": 0.75
    }
)

plt.title(
    f"Distribuição dos preços — categoria: {categoria_maior_desvio.title()}",
    fontsize=16,
    fontweight="bold",
    pad=18
)
plt.suptitle(
    "O boxplot evidencia a dispersão, a mediana e possíveis valores extremos",
    fontsize=10.5,
    y=0.93
)
plt.xlabel("Preço normal", fontsize=11, fontweight="bold")
plt.ylabel("Categoria", fontsize=11, fontweight="bold")
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# Legenda explicativa
from matplotlib.lines import Line2D
legenda_box = [
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor="#E76F51", markeredgecolor="#7A1F1F",
           markersize=7, label="Possíveis outliers")
]
plt.legend(
    handles=legenda_box,
    title="Leitura do gráfico",
    loc="upper right",
    frameon=True
)

plt.grid(axis="x", linestyle="--", alpha=0.25)
plt.tight_layout()
plt.show()


### Interpretação do boxplot

O boxplot permite observar que a categoria **lacteos** possui uma distribuição de preços bastante ampla.

Isso é relevante porque uma grande dispersão pode representar uma **estrutura de mercado heterogênea**: produtos econômicos convivendo com itens de preço elevado.

Do ponto de vista comercial, isso sugere que a categoria pode conter diferentes segmentos de consumidores. Uma estratégia única de preço ou promoção pode não ser igualmente adequada para todos os produtos.

Uma análise complementar recomendada seria cruzar esses preços com **marca, volume de vendas e margem**, caso essas variáveis estejam disponíveis.

## 8. Identificação estatística dos outliers

Para quantificar os valores extremos, utilizaremos o método do **IQR — Intervalo Interquartil**.

O método identifica observações muito afastadas do intervalo central dos dados.

É importante lembrar:

> **Outlier não significa necessariamente erro.**

Um produto pode ser um outlier porque possui características comerciais legítimas, como marca premium, embalagem maior ou posicionamento diferenciado.

In [ ]:
Q1 = dados_boxplot.quantile(0.25)
Q3 = dados_boxplot.quantile(0.75)

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1: {Q1:,.2f}")
print(f"Q3: {Q3:,.2f}")
print(f"IQR: {IQR:,.2f}")
print(f"Limite inferior: {limite_inferior:,.2f}")
print(f"Limite superior: {limite_superior:,.2f}")

In [ ]:
outliers = df[
    (df["Categoria"] == categoria_maior_desvio) &
    (
        (df["Preco_Normal"] < limite_inferior) |
        (df["Preco_Normal"] > limite_superior)
    )
].copy()

print(f"Quantidade de outliers: {len(outliers)}")

colunas_outliers = [
    coluna for coluna in ["title", "Marca", "Preco_Normal", "Categoria"]
    if coluna in outliers.columns
]

display(
    outliers[colunas_outliers]
    .sort_values("Preco_Normal", ascending=False)
)

### O que fazer com os outliers?

Foram identificados **43 outliers** em **lacteos** pelo critério do IQR.

A primeira reação não deve ser excluir esses registros. O correto é investigar sua origem.

### Possíveis perguntas de negócio

- Os produtos são realmente mais caros ou houve erro de cadastro?
- Os outliers pertencem a determinadas marcas?
- Eles representam produtos premium?
- Esses produtos também possuem descontos maiores?
- Eles possuem vendas relevantes ou são itens de baixa rotatividade?

Se os outliers forem legítimos, removê-los poderia apagar uma informação comercial importante.

**Decisão recomendada:** validar os registros antes de qualquer limpeza definitiva.

## 9. Análise dos descontos

Depois de entender o comportamento dos preços, passamos para a dimensão promocional.

A pergunta agora é:

> **Onde estão concentrados os maiores descontos médios?**

Essa métrica pode ajudar a levantar hipóteses sobre estratégias de promoção e posicionamento comercial.

Importante: a média de desconto, isoladamente, **não mede a eficiência de uma promoção**. Para avaliar eficiência seria necessário também conhecer vendas, margem, preço antes/depois e eventualmente lucro.

In [ ]:
media_desconto_categoria = (
    df.groupby("Categoria")["Desconto"]
      .mean()
      .sort_values(ascending=False)
)

display(
    media_desconto_categoria.round(2).to_frame(
        "Media_Desconto"
    )
)

### Insight comercial sobre descontos

A categoria com maior média de desconto é **congelados**, com aproximadamente **154.03**.

Por outro lado, **verduras** apresenta a menor média de desconto, aproximadamente **0.00**.

A diferença entre essas categorias pode indicar estratégias promocionais distintas.

### Mas atenção

Um desconto maior não significa automaticamente uma estratégia melhor.

Por exemplo, uma promoção pode:

- aumentar volume de vendas;
- reduzir margem;
- atrair clientes para outros produtos;
- liquidar estoque;
- ou simplesmente representar um preço promocional pouco efetivo.

**Próximo passo ideal:** cruzar desconto com volume vendido e margem. Assim seria possível avaliar não apenas "onde se dá mais desconto", mas **onde o desconto gera maior retorno comercial**.

## 10. Visualização da média de desconto por categoria

O gráfico de barras transforma a tabela anterior em uma comparação visual.

A ordenação decrescente facilita a identificação das categorias que concentram os maiores descontos médios.

In [ ]:
plt.figure(figsize=(12, 6))

ax = media_desconto_categoria.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Média de Desconto por Categoria")
plt.xlabel("Categoria")
plt.ylabel("Média de Desconto")
plt.xticks(rotation=45, ha="right")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f",
        padding=3
    )

plt.tight_layout()
plt.show()

### Como transformar esse gráfico em decisão?

O gráfico pode ser utilizado como ponto de partida para uma análise de rentabilidade.

Uma categoria com desconto médio elevado pode merecer uma investigação específica:

**Desconto alto → volume maior? → margem menor? → receita maior? → lucro maior?**

Sem essas variáveis adicionais, o gráfico mostra apenas o comportamento dos descontos, e não sua eficiência econômica.

Esse cuidado evita uma interpretação equivocada: **maior desconto ≠ maior desempenho comercial**.

## 11. Categoria × Marca

A análise por categoria pode esconder diferenças importantes entre marcas.

Duas marcas dentro da mesma categoria podem adotar políticas promocionais completamente diferentes.

Por isso, vamos calcular a média de desconto para cada combinação de:

**Categoria + Marca**

In [ ]:
categoria_marca = (
    df.groupby(["Categoria", "Marca"])
      .agg(
          Media_Desconto=("Desconto", "mean"),
          Quantidade_Produtos=("Desconto", "count")
      )
      .reset_index()
)

categoria_marca["Media_Desconto"] = (
    categoria_marca["Media_Desconto"].round(2)
)

display(
    categoria_marca
    .sort_values("Media_Desconto", ascending=False)
    .head(20)
)

### Leitura comercial por marca

Esse agrupamento permite sair de uma visão genérica de categoria e chegar a uma visão mais operacional.

Por exemplo, se uma determinada marca apresentar desconto médio muito superior às demais dentro da mesma categoria, isso pode gerar perguntas como:

- A marca possui política promocional própria?
- Os produtos têm maior margem?
- Há excesso de estoque?
- O preço normal está acima dos concorrentes?
- O desconto é usado para aumentar giro?
- O consumidor é mais sensível ao preço nessa marca?

Assim, o agrupamento **Categoria × Marca** funciona como uma ferramenta para identificar pontos que merecem investigação.

## 12. Treemap interativo

A base não possui latitude, longitude ou outra informação geográfica. Portanto, um mapa geográfico não seria apropriado.

Para preservar a ideia de uma visualização interativa, utilizaremos um **Treemap**, no qual:

- o primeiro nível representa a **Categoria**;
- o segundo nível representa a **Marca**;
- o tamanho dos blocos representa a **média de desconto**.

O usuário pode clicar nas categorias e explorar suas marcas.

In [ ]:
fig = px.treemap(
    categoria_marca,
    path=["Categoria", "Marca"],
    values="Media_Desconto",
    color="Media_Desconto",
    hover_data={
        "Media_Desconto": ":.2f",
        "Quantidade_Produtos": True
    },
    title="Média de Desconto por Categoria e Marca"
)

fig.update_layout(
    margin=dict(
        t=50,
        l=10,
        r=10,
        b=10
    )
)

fig.show()

### Como usar o Treemap para gerar hipóteses

A visualização interativa permite identificar rapidamente concentrações de desconto.

Uma área visualmente relevante pode ser investigada em seguida na tabela original.

O Treemap não deve ser interpretado como prova de que uma marca é melhor ou pior. Ele funciona como uma **ferramenta de descoberta**.

A análise seguinte deve perguntar:

> **Por que determinada marca ou categoria apresenta esse comportamento?**

Esse é o ponto em que a visualização deixa de ser apenas estética e passa a apoiar a investigação de negócio.

## 13. Ranking das combinações Categoria × Marca

Para complementar a visualização, construiremos um ranking das maiores médias de desconto.

O ranking facilita localizar exatamente os casos que merecem investigação.

In [ ]:
ranking_marcas = (
    categoria_marca
    .sort_values(
        "Media_Desconto",
        ascending=False
    )
)

display(ranking_marcas.head(20))

### Perguntas estratégicas sugeridas pelo ranking

As maiores médias de desconto podem representar oportunidades ou riscos.

**Oportunidades:**
- negociar condições comerciais;
- aumentar giro;
- testar campanhas;
- explorar elasticidade de preço.

**Riscos:**
- erosão de margem;
- dependência de promoções;
- percepção de preço elevado antes do desconto;
- descontos excessivos sem retorno proporcional.

Por isso, o ranking deve ser encarado como **ponto de partida para decisão**, e não como decisão automática.

# 14. Storytelling final — o que os dados contam?

A análise revela uma história em camadas.

### 1. Os preços não são homogêneos

As categorias apresentam diferentes níveis de preço médio e mediano. A diferença entre essas medidas mostra que algumas categorias possuem uma distribuição influenciada por valores elevados.

### 2. A dispersão é relevante

**lacteos** apresenta o maior desvio padrão, aproximadamente **3,925.82**.

Isso sugere que a categoria possui uma estrutura de preços bastante heterogênea.

### 3. Existem valores extremos

Foram identificados **43 outliers** nessa categoria pelo método do IQR.

Esses valores devem ser investigados antes de serem removidos, pois podem representar produtos premium ou diferenças legítimas de portfólio.

### 4. As estratégias de desconto também variam

**congelados** apresenta a maior média de desconto, aproximadamente **154.03**.

Isso pode indicar uma estratégia promocional mais agressiva, mas ainda não permite concluir que essa estratégia seja mais eficiente.

### 5. A análise por marca abre uma nova dimensão

O cruzamento entre categoria e marca permite localizar combinações com comportamentos promocionais diferentes.

Assim, o próximo passo natural seria cruzar essas informações com **vendas, margem, estoque e rentabilidade**.

# 15. Conclusões de negócio

## O que podemos afirmar com os dados disponíveis?

Podemos afirmar quais categorias apresentam maiores médias, medianas, dispersões e descontos médios. Também podemos identificar valores extremos e combinações de categoria e marca que merecem investigação.

## O que ainda não podemos afirmar?

Não podemos concluir, somente com essas variáveis, que:

- maior desconto gera maior venda;
- maior preço gera maior lucro;
- uma marca é mais rentável que outra;
- um outlier representa erro;
- uma estratégia promocional é eficiente.

Essas conclusões exigiriam variáveis adicionais.

## Recomendações para uma próxima análise

Se os dados estiverem disponíveis, seria interessante adicionar:

1. **Quantidade vendida** — para medir impacto sobre volume;
2. **Receita** — para medir faturamento;
3. **Custo** — para calcular margem;
4. **Estoque** — para investigar liquidação e giro;
5. **Data** — para analisar evolução temporal;
6. **Preço promocional** — para medir a diferença efetiva causada pelo desconto.

Com essas informações seria possível evoluir de uma análise descritiva para uma análise de **performance comercial e rentabilidade**.

# 16. Conclusão

Este projeto mostra que análise de dados não consiste apenas em produzir tabelas e gráficos.

O principal valor está em conectar:

**Métrica → comportamento → contexto → hipótese → decisão.**

A média e a mediana ajudam a compreender o nível central dos preços. O desvio padrão e o coeficiente de variação mostram a heterogeneidade. O boxplot e o IQR ajudam a investigar valores extremos. A análise de descontos mostra diferenças entre categorias. Finalmente, o cruzamento entre categoria e marca permite aprofundar a investigação.

### Insight final

Uma organização pode usar esse tipo de análise para descobrir **onde investigar**, mas decisões comerciais mais robustas exigem o cruzamento dessas informações com vendas, margem, estoque e comportamento ao longo do tempo.

Portanto, o resultado mais importante do projeto não é uma única categoria vencedora.

É a construção de um processo de análise capaz de transformar dados em **perguntas melhores e decisões mais fundamentadas**.

> **Dados → Informação → Insight → Ação → Aprendizado**

# 17. Reprodutibilidade e GitHub

Para a entrega final, recomenda-se organizar o repositório da seguinte maneira:

```text
projeto-final-supermercado/
│
├── projeto_final_supermercado_analise_negocios.ipynb
├── MODULO7_PROJETOFINAL_BASE_SUPERMERCADO.csv
├── README.md
└── requirements.txt
```

### Bibliotecas

```text
pandas
numpy
matplotlib
seaborn
plotly
```

O notebook pode ser publicado no GitHub juntamente com um README contendo:

- objetivo do projeto;
- descrição da base;
- principais análises;
- principais insights;
- tecnologias utilizadas;
- link para o notebook.

## 🎨 Padrão visual adotado nos gráficos

Os gráficos foram formatados com foco em **clareza, consistência e interpretação de negócio**. Foram utilizados títulos descritivos, subtítulos explicativos, rótulos de eixos, valores diretamente sobre as barras quando apropriado, grades discretas e uma paleta de cores coerente.

**Critérios de visualização:**
- cores diferentes para facilitar a comparação entre categorias;
- tons mais fortes para destacar informações relevantes, como possíveis outliers;
- ordenação dos dados quando o objetivo é criar ranking;
- títulos que comunicam a principal pergunta respondida pelo gráfico;
- legendas e subtítulos que ajudam o leitor a interpretar o que está vendo;
- redução de elementos visuais desnecessários para evitar poluição gráfica.

Esse padrão torna os resultados mais adequados para **apresentações acadêmicas, relatórios gerenciais e tomada de decisão**.